# **Data + Evaluation Stage: Baseline Cloud Detection**

**Project:** Ursa Space Systems, cloud detection in very-high-resolution (VHR) satellite imagery.
**Author:** Zainab S.
**Role this round:** Data + Eval
**October goal:** build a baseline that uses RGB brightness and saturation to find clouds and output a binary cloud mask.

## What this notebook does
My job this round is to give the team one shared test patch, so every method (RGB threshold, dark channel prior, HOT) gets scored on the exact same pixels. With reference to Rushil_Dileep_EDA.ipynb, in this notebook I:

1. Connect to the Maxar Open Data catalog
2. Pick a cloudy scene from the Belize wildfire event
3. Cut out one 1024 x 1024 patch at full resolution
4. Save it with a note of exactly where it came from
5. Turn my hand-drawn labelme labels into a ground-truth mask (the answer key)
6. Write the shared `evaluate()` function that scores any method's mask with precision, recall, and F1

**Running locally:** sections 2 to 7 need the internet (Maxar catalog) or Colab's Drive. The patch is already saved in `data/`, so to rebuild the mask just run section 1, then the Ground-truth Mask and Evaluation cells.


## Why the Belize scene?
I chose to pick the **Belize-Wildfires-June24** event because the scene had **61% cloud cover** in its metadata. A good test patch needs both cloud and clear ground, so a baseline has something to get right *and* something to get wrong.

A wildfire scene is also a fair challenge: smoke and haze can look a lot like thin cloud, and that's exactly the kind of case where a simple brightness/saturation rule might break.


# **1.Environment setup**

Importing everything needed: `numpy` and `matplotlib` for arrays and plots, `rasterio` to read pieces of the big satellite image, and `pystac` to browse the Maxar catalog.

In [30]:
# Install the geospatial stack (run once per Colab session).

import itertools
import warnings

import numpy as np
import matplotlib.pyplot as plt

import rasterio
from rasterio.windows import Window

import pystac
# import leafmap  # not used here; slow to import

# groung-truth mask
import json
from pathlib import Path
from PIL import Image, ImageDraw

warnings.filterwarnings("ignore")

# Set to True only to re-download the patch from Maxar (slow, needs internet).
# Leave False for Run All: the patch is already saved in data/.
RUN_DOWNLOAD = False


# **2. Connect to the Maxar Catalog**
Maxar's Open Data is organized as a **STAC catalog**, basically a table of contents: catalog → events (disasters) → items (individual image tiles). The code below display's the opened catalog and list some events to make sure the connection works.

In [31]:
# Needs internet: crawls the Maxar catalog
if RUN_DOWNLOAD:
    # Go through the Maxar Data STAc catalog and get the Belize Wildfire event (16%)
    CATALOG_URL = "https://maxar-opendata.s3.amazonaws.com/events/catalog.json"
    catalog = pystac.Catalog.from_file(CATALOG_URL)
    print(f"Catalog: {catalog.id}")

    events = list(catalog.get_children())
    print(f"Number of events in catalog: {len(events)}\n")
    print("Sample of available events:")
    for child in events[:20]:
        print(f"  - {child.id}")
else:
    print('Skipped (RUN_DOWNLOAD = False)')


Skipped (RUN_DOWNLOAD = False)


The catalog loaded and has 55 events. Belize-Wildfires-June24 is in the list, so I can grab it directly by name.

# **3. Find a Scene**
Each event holds many image tiles (items). I already know the ID of the tile I want, so I grab the Belize event by name and then go through its items until I hit that ID. Then I print its cloud cover and the assets (file types) it comes with.


In [32]:
# Needs internet: walks the Belize event's items (slow)
if RUN_DOWNLOAD:
    #First pull the event out by name, then walk its items until you hit your scene's id.
    # I'm assuming you meant the 61% scene, since the "16" is just the start of its id.

    # Grab the Belize Wildefire event by the name
    belize = catalog.get_child("Belize-Wildfires-June24")
    print(f"Event: {belize.id}")

    # I need to walk it's item and stop at my interested scene's ID (61%)
    scene_id = "16/033131012101/1040010051B60600"
    scene = next(
        item for item in belize.get_items(recursive=True)
        if item.id == scene_id
    )
    print (f"Scene: {scene_id}")
    print(f"Cloud cover: {scene.properties.get('tile:clouds_percent')}%")
    print(f"Assets:", list(scene.assets.keys()))
else:
    print('Skipped (RUN_DOWNLOAD = False)')


Skipped (RUN_DOWNLOAD = False)


The scene has **61% cloud cover** and four assets. I only need `visual`, the 8-bit RGB version of the image (values 0 to 255), since our baseline works on RGB brightness and saturation.

# **4. Get the image link**
The catalog stores file paths *relative* to each item, so `rasterio` can't open them as-is. This small function turns the `visual` asset into a full web link (URL) that I can read from directly.


In [33]:
# Needs the scene from the cell above
if RUN_DOWNLOAD:
    def get_visual_href(scene):
        """Return the ABSOLUTE href of the 8-bit RGB 'visual' COG, with sensible fallbacks.

        Maxar STAC stores asset hrefs relative to each item's JSON, so we resolve them
        with get_absolute_href() -- otherwise rasterio.open() gets a bare filename.
        """
        for key in ("visual", "rgb", "B3B2B1"):
            if key in scene.assets:
                return scene.assets[key].get_absolute_href()
        # Last resort: first available asset.
        return next(iter(scene.assets.values())).get_absolute_href()

    # Print image link
    href = get_visual_href(scene)   # scene = the item I found in the earlier cell
    print(href)
else:
    print('Skipped (RUN_DOWNLOAD = False)')


Skipped (RUN_DOWNLOAD = False)


# **5. Read One 1024 x 1024 Window**

The full image is **17,408 x 17,408 pixels**, which is way too big to load and label by hand. So I only read one 1024 x 1024 square at full detail.

How I picked the spot: I looked at a shrunk-down overview of the whole tile and chose a top-left corner on a cloud edge, at **row 3300, col 8500**.

- `row_off` moves the window **down** (bigger = lower)
- `col_off` moves the window **right** (bigger = further right)

*Lesson learned:* I first swapped these two and landed on the black "no data" area outside the image. The order matters.

The stretched copy (`view`) is only for display. The real `patch` keeps its original 0 to 255 values, and that's the version we evaluate on.


In [34]:
# Needs internet: reads the window from the remote image
if RUN_DOWNLOAD:
    tile = 1024

    row_off = 3300
    col_off = 8500

    # use rasterio to open that image as src, mark it's width and height (as well as it's band)
    with rasterio.open(href) as src:
      H, W = src.height, src.width
      print(f"Full image: {W} x {H} pixels, {src.count} bands")
      window = Window(col_off, row_off, tile, tile)

      #We need to read just that square -> shape (3, 1024, 1024), values 0-255
      patch = src.read([1, 2, 3], window=window)

      #We then need to rearrange to (height, width, 3), the usual image shape
      patch = np.transpose(patch, (1, 2, 0))
      print(f"Window starts at row {row_off}, col {col_off}")
      print(f"Patch shape: {patch.shape}, type: {patch.dtype}")

      #Shows it -> (stretched copy only for viewing, the real patch stays untouched)
      view = patch.astype(float)
      lo, hi = np.percentile(view, (2, 98))
      view = np.clip((view - lo) / (hi - lo), 0, 1)

      # View everything
      plt.figure(figsize=(7, 7))
      plt.imshow(view)
      plt.title("Belize Wildfires, 1024x1024 window")
      plt.axis("off")
      plt.show()
else:
    print('Skipped (RUN_DOWNLOAD = False)')


Skipped (RUN_DOWNLOAD = False)


# **6. Looking at the patch**
This patch works well for evaluation -> about **half cloud (left) and half forest (right)**, with a **soft, hazy edge** between them.

- The clear cloud and clear forest should be easy for any method.
- The hazy edge is likely going to be the hard part. It's where I expect the RGB threshold to struggle, and where comparing methods will actually be interesting.

**Question:** does thin haze count as cloud in our ground truth? We need to agree before labeling.

**No data warning:** black areas outside the image footprint are *not* clear ground. They should be excluded from evaluation, never counted as "no cloud." (This patch sits inside the valid area, but other patches might not.)


# **7. Save The Patch**

Save the **real** patch (not the stretched view) to Google Drive as a PNG, plus a small text file recording the scene ID, image link, and window offsets. That way anyone on the team can reload the exact same pixels, and we always know where the patch came from.

The file name `belize_r11384_c1536` encodes the row and column offsets, so it's traceable at a glance.

*Note: Drive has to be mounted first (`from google.colab import drive; drive.mount('/content/drive')`). Colab's own files disappear when the session ends.*


In [35]:
# Saves the downloaded patch locally
if RUN_DOWNLOAD:
    import os
    from PIL import Image

    # Folder in your Drive for the eval set
    save_dir = "../data"   # was Colab Drive: /content/drive/MyDrive/ursa_eval
    os.makedirs(save_dir, exist_ok=True)

    # Name the file after the scene and window so it's easy to trace
    name = f"belize_r{row_off}_c{col_off}"

    # Save the REAL patch (not the stretched view)
    Image.fromarray(patch).save(f"{save_dir}/{name}.png")

    # Save a small note of where it came from
    with open(f"{save_dir}/{name}.txt", "w") as f:
        f.write(f"scene: {scene_id}\n")
        f.write(f"href: {href}\n")
        f.write(f"row_off: {row_off}, col_off: {col_off}, size: {tile}\n")

    print("Saved:", f"{save_dir}/{name}.png")
else:
    print('Skipped (RUN_DOWNLOAD = False)')


Skipped (RUN_DOWNLOAD = False)


# **Ground-truth Mask (The Answer)**
**Evaluation set:** a few image patches plus their correct answers, called ground-truth masks.
Every method is scored against the same set, and the set never changes.

**Our first patch:** `belize_r11384_c1536.png`
(Belize-Wildfires-June24, 1024 x 1024 window at row_off 3300, col_off 8500)

**How the answer key will be made:**
- We trace the clouds by hand in labelme, not with a formula.
  If a brightness rule drew the mask, a brightness-based method would be graded against itself and score too high.
- **cloud** = clear cloud. These pixels become 1.
- **unsure** = the thin hazy edge. These pixels are left out of scoring.
- Everything else is ground and becomes 0.
- Nodata pixels (all bands 0) are also left out of scoring.

**Output:**
- `belize_r11384_c1536_mask.png`: 1 = cloud, 0 = not cloud
- `belize_r11384_c1536_valid.npy`: True = scored, False = skipped (unsure or nodata)

Both files are saved in `data/masks/` (small enough for GitHub), with a copy in `Ursa1B Shared Drive/01_Data/masks/`.





In [ ]:
# Find the data folder (works from repo root or notebooks/)
DATA = Path("data") if Path("data/belize_r11384_c1536.png").exists() else Path("../data")

# Load the patch and the labelme labels
img = np.array(Image.open(DATA / "belize_r11384_c1536.png").convert("RGB"))
ann = json.load(open(DATA / "belize_r11384_c1536.json"))
H, W = img.shape[:2]


# Pull out cloud, clear, unsure and turn them into numbers
def draw(label):
    #True inside every polygon with this label.
    canvas = Image.new("L", (W, H), 0)
    for s in ann["shapes"]:
        if s["label"] == label:
            ImageDraw.Draw(canvas).polygon([tuple(p) for p in s["points"]], fill=1)
    return np.array(canvas, dtype=bool)

cloud, clear, unsure, nodata = draw("cloud"), draw("clear"), draw("unsure"), draw("nodata")

# 1 = cloud and 0 = no cloud
true_mask = (cloud & ~clear).astype(np.uint8)
# False =  don;t score these pixels
valid = ~(unsure | nodata)

# Save
(DATA / "masks").mkdir(exist_ok=True)
Image.fromarray(true_mask * 255).save(DATA / "masks/belize_r11384_c1536_mask.png")
np.save(DATA / "masks/belize_r11384_c1536_valid.npy", valid)

Evaluation function

In [37]:
# Shared evaluation lives in src/eval.py so the whole team uses the same code
import sys
sys.path.append("..")              # let the notebook see the repo root
from src.eval import evaluate       # the shared team function

# Sanity checks
print("shape:", true_mask.shape, "values:", np.unique(true_mask))
print("cloud share:", round(true_mask.mean(), 2), "scored share:", round(valid.mean(), 2))
print("self-test (should be all 1.0):", evaluate(true_mask, true_mask, valid))


shape: (1024, 1024) values: [0 1]
cloud share: 0.54 scored share: 0.91
self-test (should be all 1.0): {'precision': 1.0, 'recall': 1.0, 'f1': 1.0}


# **8. Status and next steps**
Rushil writes `detect(image, b, s) -> mask` for the RGB brightness/saturation baseline, and we score it with `evaluate(pred, true_mask, valid)`.

**Quick metric reminder:**
- *Precision:* of the pixels we called cloud, how many really are cloud?
- *Recall:* of the real cloud pixels, how many did we catch?
- *F1:* one number that balances the two.

# **Additional: Metrics right now as we move onto next steps**

In [ ]:
DATA = Path("data") if Path("data/belize_r11384_c1536.png").exists() else Path("../data")

mask = np.array(Image.open(DATA / "masks/belize_r11384_c1536_mask.png")) > 0
valid = np.load(DATA / "masks/belize_r11384_c1536_valid.npy")
ann = json.load(open(DATA / "belize_r11384_c1536.json"))

total = mask.size
print("Pixels labeled:", total)
print("Valid (scored):", valid.sum(), f"({valid.mean():.1%})")
print("Excluded (unsure/nodata):", f"{1 - valid.mean():.1%}")
print("Cloud % of valid:", f"{mask[valid].mean():.1%}")
print("Polygons:", len(ann["shapes"]),
      "| Vertices:", sum(len(s["points"]) for s in ann["shapes"]))

Pixels labeled: 1048576
Valid (scored): 955202 (91.1%)
Excluded (unsure/nodata): 8.9%
Cloud % of valid: 59.4%
Polygons: 3 | Vertices: 155
